# ProjectOwl — Step-by-Step Walkthrough

This notebook demonstrates each component of the pipeline with visible outputs.

**Prerequisites:** PostgreSQL running, `.env` with API keys, `pip install -r requirements.txt && pip install -e .`

## 0. Setup & Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Avoid OpenMP conflict (PyTorch vs numpy/scipy)

import sys
from pathlib import Path

# Add project root so we can import owl
ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Config — Tunable Parameters

All scales, frequencies, and hyperparameters live in one place.

In [ ]:
from owl import config

print("=== Key config values ===")
print(f"Price frequency:     {config.PRICE_FREQUENCY} (multiplier={config.PRICE_MULTIPLIER})")
print(f"Input window:        {config.INPUT_WINDOW_MINUTES} bars")
print(f"Prediction window:   {config.PREDICTION_WINDOW_MINUTES} bars")
print(f"Categories:          {config.NUM_CATEGORIES} (derived, no pre-set thresholds)")
print(f"Category names:      {config.CATEGORY_NAMES}")
print(f"MA windows:          {config.MA_WINDOWS}")
print(f"Normalization:       {config.NORMALIZATION_METHOD}")
print(f"Database URL:        {config.DATABASE_URL[:30]}...")

## 2. Database — Create Tables

In [ ]:
from owl.data.db import create_tables

create_tables()
print("Tables created: training_cases, validation_cases, case_metadata, training_metrics")

## 3. Data Query Engine — Fetch from APIs

**3a.** Massive API — daily OHLCV bars

In [ ]:
from owl.data.massive_client import MassiveClient
from owl.config import PRICE_FREQUENCY, PRICE_MULTIPLIER

client = MassiveClient()
df = client.get_aggs("AAPL", PRICE_MULTIPLIER, PRICE_FREQUENCY, "2024-01-01", "2024-01-31")

print(f"Shape: {df.shape}")
display(df.head(10))

**3b.** SHARADAR — daily valuation metrics

In [ ]:
from owl.data.sharadar_client import get_daily_metrics

daily = get_daily_metrics("AAPL", "2023-12-20", "2024-01-31")
print(f"Shape: {daily.shape}")
display(daily.head(10))

**3c.** SHARADAR SF1 — quarterly fundamentals

In [ ]:
from owl.data.sharadar_client import get_fundamentals

sf1 = get_fundamentals("AAPL", start_date="2023-01-01", end_date="2024-01-31")
print(f"Shape: {sf1.shape}")
display(sf1.head(10))

**3d.** Full merge — fetch and merge one case, then save

In [ ]:
from datetime import date
from owl.data.query_engine import fetch_and_merge
from owl.data.db import insert_case_data

merged = fetch_and_merge(
    symbol="AAPL",
    start_date=date(2024, 1, 2),
    end_date=date(2024, 3, 15),
    case_id="demo_aapl_20240102",
)

if merged is not None:
    print(f"Merged shape: {merged.shape}")
    print(f"Columns: {list(merged.columns)}")
    display(merged[["timestamp", "open", "high", "low", "close", "volume", "marketcap", "pe"]].head(15))
else:
    print("No data returned — check API keys and date range.")

**3e.** Populate DB with random cases (training + validation)

In [ ]:
from owl.data.query_engine import populate_database

populate_database(n_train=5, n_val=2, tickers=["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"])
print("Done populating.")

## 4. Data Feeder — PyTorch Dataset

In [ ]:
from owl.preprocessing.pipeline import PreprocessingPipeline
from owl.data.feeder import OwlDataset
from owl.config import TRAINING_TABLE

pipeline = PreprocessingPipeline()
dataset = OwlDataset(TRAINING_TABLE, pipeline=pipeline)

print(f"Total windows: {len(dataset)}")
print(f"Feature dim:   {dataset.get_feature_dim()}")

# Get one sample
X, y = dataset[0]
print(f"\nSample input shape: {X.shape}  (T={X.shape[0]}, F={X.shape[1]})")
print(f"Sample label (category): {y}")

## 5. Preprocessing — Pipeline Output

In [ ]:
from owl.data.db import load_case_data
from owl.config import TRAINING_TABLE
from owl.data.db import get_case_ids

case_ids = get_case_ids(TRAINING_TABLE)
if case_ids:
    raw = load_case_data(case_ids[0], TRAINING_TABLE)
    pipeline = PreprocessingPipeline()
    processed = pipeline.transform(raw)
    
    print("Feature columns (first 20):")
    print(pipeline.feature_columns[:20])
    print(f"\nProcessed shape: {processed.shape}")
    display(processed[pipeline.feature_columns[:8]].head(10))
else:
    print("No cases in DB — run populate_database first.")

## 6. Model — CNN Forward Pass

In [ ]:
import torch
from owl.models.cnn_model import TimeSeriesCNN

in_features = dataset.get_feature_dim()
model = TimeSeriesCNN(in_features)

X_batch = torch.from_numpy(X.numpy() if hasattr(X, "numpy") else X).unsqueeze(0).float()
logits = model(X_batch)
probs = torch.softmax(logits, dim=1)

print(f"Input shape:  {X_batch.shape}")
print(f"Logits:      {logits.detach().numpy().ravel()}")
print(f"Probs:       {probs.detach().numpy().ravel()}")
print(f"Predicted:   {logits.argmax(1).item()}")
print(f"Categories:  {config.CATEGORY_NAMES}")

## 7. Training — Short Run

In [ ]:
from torch.utils.data import DataLoader
from owl.data.feeder import make_dataloaders
from owl.models.cnn_model import CNNTrainer

train_loader, val_loader = make_dataloaders(pipeline=pipeline, batch_size=16, num_workers=0)
trainer = CNNTrainer(train_loader.dataset.get_feature_dim())

history = trainer.fit(train_loader, val_loader, epochs=3)
print("\nHistory keys:", list(history.keys()))

## 8. Reports — Training Curves & Feature Importance

In [ ]:
from owl.visualization.reports import plot_training_history, plot_feature_importance
from owl.config import REPORT_DIR

report_dir = REPORT_DIR / "cnn"
report_dir.mkdir(parents=True, exist_ok=True)

fig1 = plot_training_history(history, title="CNN Training")
plt.show()

sample_batch, _ = next(iter(val_loader))
fig2 = plot_feature_importance(trainer.model, sample_batch, pipeline.feature_columns, device=trainer.device)
plt.show()

## 9. t-SNE — Latent Space Visualisation

In [ ]:
from owl.models.tsne_viz import compute_tsne, plot_tsne_2d

latents, labels = trainer.extract_latents(val_loader)
print(f"Latents shape: {latents.shape}")

if len(latents) >= 10:
    emb_2d = compute_tsne(latents, n_components=2)
    fig = plot_tsne_2d(emb_2d, labels, title="CNN t-SNE 2D")
    plt.show()
else:
    print("Too few samples for t-SNE (need ~10+). Populate more data.")

## 10. Category Examples — Time Series per Class

In [ ]:
from owl.visualization.reports import plot_category_examples

fig = plot_category_examples(val_loader.dataset, save_path=report_dir / "category_examples.png")
plt.show()